# Stage 2 — Retrieval-Augmented Distractor Generation (minimal)

End-to-end: **frozen Stage 1 retrieval → prompt construction → LLM generation → JSONL**.

| Setting | Value |
|---|---|
| Retriever | MNRL fine-tuned MiniLM (**frozen**, Variant B deployment query) |
| Query | Subject + Construct + Question + Correct Answer |
| Exemplars | k = 5, misconception labels **shown** |
| Generator | Qwen2.5-7B-Instruct (4-bit) |
| Candidates | M = 5 per question |
| Targets | 187 question-level targets (431 test QDPs collapsed by question) |

Stage 1 is used strictly read-only. Not in scope here: difficulty control, Stage 3 re-ranking, evaluation, ablations.

**Set `REPO_URL`, Runtime → GPU, then Run all.** Generation appends to JSONL after every question, so a disconnect loses nothing — just re-run and it resumes.

In [ ]:
# --- The ONLY cell you edit ---
REPO_URL = "https://github.com/YOUR_USERNAME/distractor.git"  # <-- set this

# Path to a saved MNRL retriever, if you have one (skips the ~15 min re-train).
EXISTING_RETRIEVER = None   # e.g. "/content/drive/MyDrive/finetuned_mnrl"
SEED = 42
K = 5      # retrieved exemplars per prompt
M = 5      # candidate distractors per question

In [ ]:
# --- Verify GPU ---
import torch
!nvidia-smi -L
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > GPU"
print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
# --- Clone repo and install dependencies ---
import os
if not os.path.exists("distractor"):
    !git clone {REPO_URL} distractor
%cd distractor
%pip install -q -r requirements_gpu.txt
assert os.path.exists("datasets/train.csv"), "datasets/train.csv missing from repo"
print("Dataset present.")

In [ ]:
# --- Prepare splits (identical to every prior stage) ---
import os
if not os.path.exists("outputs/results/train_qdp.csv"):
    !python scripts/01_prepare_dataset.py
if not os.path.exists("outputs/triplets/train_triplets.jsonl"):
    !python scripts/03_create_triplets.py
print("Splits ready.")

In [ ]:
# --- Obtain the frozen Stage 1 retriever ---
# Reproduces the seed-42 MNRL model if none is supplied. This is a
# REPRODUCTION of the frozen configuration, not a new training condition.
import os
RETRIEVER = EXISTING_RETRIEVER or "outputs/models/finetuned_mnrl"
if EXISTING_RETRIEVER is None and not os.path.exists(f"{RETRIEVER}/config.json"):
    !python scripts/train_gpu.py --objective mnrl --seed {SEED} \
      --save-steps 50 --eval-steps 50 --keep-all-checkpoints \
      --output {RETRIEVER}
else:
    print(f"Using existing retriever: {RETRIEVER}")
print("Retriever ready:", RETRIEVER)

## Step 1 — Inspect a prompt before spending GPU time

`--dry-run` performs retrieval and builds prompts without loading the LLM. Check that the exemplars are sensible, the target block is right, and no exemplar comes from the target question itself.

In [ ]:
!python scripts/08_generate_distractors.py \
  --retriever {RETRIEVER} --k {K} --m {M} --seed {SEED} \
  --limit 1 --dry-run

## Step 2 — Smoke test (5 questions, real generation)

Loads the LLM and generates for 5 questions. Confirms the model loads in 4-bit, output parses, and records are written — before committing to the full run. Writes to a separate `_smoke.jsonl` so it never mixes with the real output.

In [ ]:
!python scripts/08_generate_distractors.py \
  --retriever {RETRIEVER} --k {K} --m {M} --seed {SEED} --limit 5

In [ ]:
# --- Inspect one smoke-test result ---
import json
rec = [json.loads(l) for l in open("outputs/generation/stage2_generations_smoke.jsonl")][0]
print("QUESTION:", rec["question_text"][:300])
print("CORRECT :", rec["correct_answer"])
print(f"\nExemplars retrieved: {len(rec['retrieved'])} "
      f"(misconception-matched: {rec['n_exemplars_matched']})")
print("parse status:", rec["parse_status"])
print("\n--- GENERATED CANDIDATES ---")
for i, c in enumerate(rec["candidates"], 1):
    print(f"{i}. {c['distractor']}")
    print(f"   intended misconception: {c['stated_misconception']}")
print("\n--- GOLD (reference, not shown to the model) ---")
for g in rec["gold_distractors"]:
    print(f" * {g['distractor']}  <- {g['misconception_name']}")

## Step 3 — Full run (187 questions)

Appends to `outputs/generation/stage2_generations.jsonl` after each question. **If Colab disconnects, just re-run this cell** — completed questions are skipped automatically.

In [ ]:
!python scripts/08_generate_distractors.py \
  --retriever {RETRIEVER} --k {K} --m {M} --seed {SEED}

In [ ]:
# --- Run summary ---
import json, pandas as pd
recs = [json.loads(l) for l in open("outputs/generation/stage2_generations.jsonl")]
print(f"questions generated : {len(recs)} / 187")
print(f"parse status        : {pd.Series([r['parse_status'] for r in recs]).value_counts().to_dict()}")
print(f"mean candidates     : {sum(len(r['candidates']) for r in recs)/max(len(recs),1):.2f}")
print(f"questions with >=1 misconception-matched exemplar: "
      f"{sum(r['n_exemplars_matched'] > 0 for r in recs)}/{len(recs)} "
      f"({sum(r['n_exemplars_matched'] > 0 for r in recs)/max(len(recs),1):.1%})")
print("\nNote: the matched-exemplar rate above is a property of retrieval at k=5,")
print("not of generation. It is recorded per question so later evaluation can test")
print("whether retrieval quality predicts generation quality.")

In [ ]:
# --- Package and download ---
!zip -qr stage2_generations.zip outputs/generation
from google.colab import files
files.download("stage2_generations.zip")